In [4]:
import numpy as np
import pandas as pd
import os
from tensorflow import keras
from tensorflow.keras import layers

In [18]:
# Load tabular dataset for shallow feedforward DNN
folder_path = r'./data/processed'
path_main = os.path.join(folder_path, 'dataset_with_returns.csv')

print("Cargando dataset principal...")
df = pd.read_csv(path_main)

Cargando dataset principal...


In [8]:
df.head()

,R_-239,R_-238,R_-237,R_-236,R_-235,R_-234,R_-233,R_-232,R_-231,R_-230,...,R_-6,R_-5,R_-4,R_-3,R_-2,R_-1,R_0,ticker,date,target
0,-0.010648,0.000448,0.002914,-0.003799,-0.052266,0.017988,-0.017903,-0.014441,0.004083,-0.012201,...,0.016121,0.007682,0.002651,0.003471,0.002306,0.001315,-0.023962,A,2014-01-23,0
1,0.000448,0.002914,-0.003799,-0.052266,0.017988,-0.017903,-0.014441,0.004083,-0.012201,-0.007750,...,0.007682,0.002651,0.003471,0.002306,0.001315,-0.023962,-0.026904,A,2014-01-24,1
2,0.002914,-0.003799,-0.052266,0.017988,-0.017903,-0.014441,0.004083,-0.012201,-0.007750,0.018550,...,0.002651,0.003471,0.002306,0.001315,-0.023962,-0.026904,0.007431,A,2014-01-27,-1
3,-0.003799,-0.052266,0.017988,-0.017903,-0.014441,0.004083,-0.012201,-0.007750,0.018550,-0.005991,...,0.003471,0.002306,0.001315,-0.023962,-0.026904,0.007431,-0.003430,A,2014-01-28,0
4,-0.052266,0.017988,-0.017903,-0.014441,0.004083,-0.012201,-0.007750,0.018550,-0.005991,0.010848,...,0.002306,0.001315,-0.023962,-0.026904,0.007431,-0.003430,-0.007746,A,2014-01-29,1


## Split in train and test

In [19]:
def temporal_train_test_split(df, train_ratio=0.75):
    train_dfs = []
    test_dfs = []

    # Per-ticker temporal split when ticker is available
    if "ticker" in df.columns:
        df = df.copy()
        df["time_idx"] = df.groupby("ticker").cumcount()

        for ticker in df["ticker"].unique():
            df_t = df[df["ticker"] == ticker].sort_values("time_idx")
            n = len(df_t)
            train_size = int(train_ratio * n)

            train_dfs.append(df_t.iloc[:train_size])
            test_dfs.append(df_t.iloc[train_size:])

        df_train = pd.concat(train_dfs).reset_index(drop=True)
        df_test = pd.concat(test_dfs).reset_index(drop=True)
        return df_train, df_test

    # Fallback temporal split by date/index when ticker is not available
    df_sorted = df.copy()
    if "date" in df_sorted.columns:
        df_sorted["date"] = pd.to_datetime(df_sorted["date"], errors="coerce")
        df_sorted = df_sorted.sort_values("date")

    split_idx = int(train_ratio * len(df_sorted))
    df_train = df_sorted.iloc[:split_idx].reset_index(drop=True)
    df_test = df_sorted.iloc[split_idx:].reset_index(drop=True)
    return df_train, df_test

In [20]:
df_train, df_test = temporal_train_test_split(df)

In [27]:
print(df_train.shape)
print(df_test.shape)

if "ticker" in df_train.columns:
    print(df_train.groupby("ticker").size().head())
    print(df_test.groupby("ticker").size().head())
else:
    print("'ticker' column not present; temporal split used date/index fallback.")

(463901, 9)
(154634, 9)
'ticker' column not present; temporal split used date/index fallback.


In [37]:
# Quick target sanity check
print("y_train_eval min/max:", float(np.min(y_train_eval)), float(np.max(y_train_eval)))
print("y_val_eval min/max:", float(np.min(y_val_eval)), float(np.max(y_val_eval)))
print("Unique y values in validation (first 20):", np.unique(y_val_eval)[:20])
print("Count y==0:", int(np.sum(y_val_eval == 0)))
print("Count y==1:", int(np.sum(y_val_eval == 1)))

y_train_eval min/max: -1.0 1.0
y_val_eval min/max: -1.0 1.0
Unique y values in validation (first 20): [-1.  0.  1.]
Count y==0: 39685
Count y==1: 29939


In [ ]:
# ### Training
# Build and train a shallow network (single-layer logistic model)

# Resolve source arrays from available notebook variables
if "X_np" in globals() and "y_np" in globals():
    X_all = np.asarray(X_np, dtype=np.float32)
    y_all = np.asarray(y_np, dtype=np.float32).reshape(-1)
elif "X_train_np" in globals() and "y_train_np" in globals():
    X_all = np.asarray(X_train_np, dtype=np.float32)
    y_all = np.asarray(y_train_np, dtype=np.float32).reshape(-1)
else:
    raise ValueError("No se encontraron arrays base. Ejecuta primero la preparacion de datos.")

# Keep exactly 31 features
if X_all.ndim != 2:
    raise ValueError(f"X debe ser 2D. Shape recibido: {X_all.shape}")
if X_all.shape[1] < 31:
    raise ValueError(f"Se requieren al menos 31 features y se encontraron {X_all.shape[1]}")
X_all = X_all[:, :31]

# Convert target to binary: BUY(1) vs NOT-BUY(0)
y_bin = (y_all > 0).astype(np.float32)

# Holdout split (80/20)
rng = np.random.default_rng(42)
idx = np.arange(len(X_all))
rng.shuffle(idx)
cut = int(0.8 * len(idx))
train_idx = idx[:cut]
val_idx = idx[cut:]

X_train_shallow = X_all[train_idx]
y_train_shallow = y_bin[train_idx]
X_val_shallow = X_all[val_idx]
y_val_shallow = y_bin[val_idx]

# Define shallow model
shallow_model = keras.Sequential([
    layers.Input(shape=(31,)),
    layers.Dense(1, activation="sigmoid")
])

shallow_model.compile(
    loss="binary_crossentropy",
    optimizer=keras.optimizers.Adam(),
    metrics=["accuracy"]
)

history_shallow = shallow_model.fit(
    X_train_shallow,
    y_train_shallow,
    validation_data=(X_val_shallow, y_val_shallow),
    epochs=8,
    batch_size=128,
    verbose=1
)

print("Training completed.")

In [ ]:
# ## Validation
# Evaluate the trained shallow model on holdout validation data

if "shallow_model" not in globals():
    raise ValueError("No existe shallow_model. Ejecuta primero la celda de Training.")

val_loss_shallow, val_acc_shallow = shallow_model.evaluate(
    X_val_shallow,
    y_val_shallow,
    verbose=0
)

y_prob_shallow = shallow_model.predict(X_val_shallow, verbose=0).reshape(-1)
y_pred_shallow = (y_prob_shallow >= 0.5).astype(np.int32)
y_true_shallow = y_val_shallow.astype(np.int32)

print(f"Validation Loss: {val_loss_shallow:.4f}")
print(f"Validation Accuracy: {val_acc_shallow:.4f}")
print("Validation completed.")

In [ ]:
# ## Metrics
# Compute extra metrics for the shallow classifier

tp = int(np.sum((y_pred_shallow == 1) & (y_true_shallow == 1)))
tn = int(np.sum((y_pred_shallow == 0) & (y_true_shallow == 0)))
fp = int(np.sum((y_pred_shallow == 1) & (y_true_shallow == 0)))
fn = int(np.sum((y_pred_shallow == 0) & (y_true_shallow == 1)))

precision_shallow = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall_shallow = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1_shallow = (
    2 * precision_shallow * recall_shallow / (precision_shallow + recall_shallow)
    if (precision_shallow + recall_shallow) > 0
    else 0.0
)

auc_metric_shallow = keras.metrics.AUC()
auc_metric_shallow.update_state(y_true_shallow, y_prob_shallow)
auc_shallow = float(auc_metric_shallow.result().numpy())

print(f"Precision: {precision_shallow:.4f}")
print(f"Recall: {recall_shallow:.4f}")
print(f"F1: {f1_shallow:.4f}")
print(f"AUC: {auc_shallow:.4f}")
print(f"Confusion Matrix -> TN: {tn}, FP: {fp}, FN: {fn}, TP: {tp}")